# 11 — RAG Baseline: Few-Shot Retrieval vs Fine-Tuning

**Goal**: Evaluate a RAG approach (retrieve similar training examples → few-shot prompt → base Llama-3) as a baseline to compare against our fine-tuned model.

**Method**: For each test question, we:
1. Embed the question + schema with `all-MiniLM-L6-v2`
2. Retrieve top-3 similar training examples from ChromaDB
3. Build a few-shot prompt with those examples as demonstrations
4. Send to **base** Llama-3-8B-Instruct (no fine-tuning, no chat prefix)
5. Evaluate with our standard execution accuracy pipeline

**Runtime**: Google Colab T4 GPU (~1-2 hours for 500 examples)

## 0. Setup

In [ ]:
# Install dependencies
!pip install -q transformers peft bitsandbytes accelerate datasets torch
!pip install -q chromadb sentence-transformers langchain-community

In [ ]:
# Clone the repo (or mount Google Drive if preferred)
!git clone https://github.com/oLittle-five/text-to-sql-llama.git
%cd text-to-sql-llama

In [ ]:
import json
import time
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

## 1. Build ChromaDB Index on Colab

We regenerate the training JSONL and build the vector index directly on Colab.
This avoids uploading the ~200MB ChromaDB folder.

In [ ]:
# Generate training JSONL from WikiSQL
from src.data.prepare_dataset import prepare_and_save
prepare_and_save("data/processed")

In [ ]:
# Build the ChromaDB index (~5-10 min on Colab)
from src.rag.build_index import build_index
build_index("data/processed/train.jsonl", "data/chroma_db")

In [ ]:
# Initialize the RAG pipeline
from src.rag.rag_pipeline import RAGPipeline
rag = RAGPipeline(db_dir="data/chroma_db", top_k=3)

## 2. Quick Retrieval Sanity Check

Verify that retrieved examples look relevant before running full evaluation.

In [ ]:
# Load test set
ds = load_dataset("Salesforce/wikisql", trust_remote_code=True)
test_examples = list(ds["test"].select(range(500)))
print(f"Test examples: {len(test_examples)}")

In [ ]:
# Preview: retrieve examples for the first test question
ex = test_examples[0]
print(f"Test question: {ex['question']}")
print(f"Test columns:  {ex['table']['header']}")
print()

retrieved = rag.retrieve(ex["question"], ex["table"]["header"], ex["table"]["types"])
for i, r in enumerate(retrieved, 1):
    print(f"--- Example {i} (distance: {r['distance']:.4f}) ---")
    print(r["text"][:200])
    print()

In [ ]:
# Preview the full few-shot prompt for the first test example
prompt = rag.build_few_shot_prompt(
    ex["question"], ex["table"]["header"], ex["table"]["types"]
)
print(prompt)
print(f"\nPrompt length: {len(prompt)} chars")

## 3. Load Base Model

In [ ]:
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
print("Model loaded.")

In [ ]:
# Generation config for base model (NO repetition_penalty — same as notebook 06)
GENERATION_CONFIG = dict(
    max_new_tokens=128,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=[
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    ],
)

## 4. SQL Generation with RAG

In [ ]:
import re

def extract_sql(response: str, columns: list[str]) -> str:
    """
    Extract the SQL query from the model's response.
    Takes the first line that looks like SQL, fixes common issues.
    """
    # Take just the first line (model may ramble after the SQL)
    sql = response.strip().split("\n")[0].strip()
    
    # Remove markdown code fences if present
    sql = sql.replace("```sql", "").replace("```", "").strip()
    
    # Fix column names: ensure backtick quoting for columns with special chars
    for col in columns:
        if col in sql and f"`{col}`" not in sql:
            # Only quote if column has special characters
            if any(c in col for c in " /()-+."):
                sql = sql.replace(col, f"`{col}`")
    
    return sql


def generate_sql_rag(model, tokenizer, rag_pipeline, question, columns, types, gen_config):
    """
    Generate SQL using RAG few-shot prompting.
    """
    # Build few-shot prompt via retrieval
    prompt = rag_pipeline.build_few_shot_prompt(question, columns, types)
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_config)
    
    # Decode only the new tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    return extract_sql(response, columns)

In [ ]:
# Quick test: generate SQL for the first example
ex = test_examples[0]
pred_sql = generate_sql_rag(
    model, tokenizer, rag,
    ex["question"], ex["table"]["header"], ex["table"]["types"],
    GENERATION_CONFIG,
)
print(f"Question:  {ex['question']}")
print(f"Predicted: {pred_sql}")

## 5. Run Full Evaluation (500 examples)

In [ ]:
N_EXAMPLES = 500

predictions = []
start_time = time.time()

for i, ex in enumerate(tqdm(test_examples[:N_EXAMPLES], desc="RAG inference")):
    pred_sql = generate_sql_rag(
        model, tokenizer, rag,
        ex["question"], ex["table"]["header"], ex["table"]["types"],
        GENERATION_CONFIG,
    )
    predictions.append(pred_sql)
    
    # Progress check every 50 examples
    if (i + 1) % 50 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (N_EXAMPLES - i - 1) / rate
        print(f"  [{i+1}/{N_EXAMPLES}] {elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining")

total_time = time.time() - start_time
print(f"\nDone! {N_EXAMPLES} examples in {total_time:.0f}s ({total_time/N_EXAMPLES:.1f}s/example)")

## 6. Evaluate with Execution Accuracy

In [ ]:
from src.data.prepare_dataset import build_sql_string
from src.data.sql_executor import execute_sql

def normalize_result(result):
    """Normalize SQL result for comparison."""
    if result is None:
        return None
    return sorted([tuple(str(v).lower().strip() for v in row) for row in result])


def evaluate(predictions, examples):
    """Compute execution accuracy."""
    total = len(predictions)
    correct = syntax_errors = wrong_result = 0
    
    for pred, ex in zip(predictions, examples):
        table = ex["table"]
        gold_sql = build_sql_string(ex["sql"], table["header"], table["types"])
        
        pred_results, pred_error = execute_sql(table, pred)
        gold_results, gold_error = execute_sql(table, gold_sql)
        
        if pred_error:
            syntax_errors += 1
        elif normalize_result(pred_results) == normalize_result(gold_results):
            correct += 1
        else:
            wrong_result += 1
    
    return {
        "execution_accuracy": correct / total,
        "syntax_error_rate": syntax_errors / total,
        "wrong_result_rate": wrong_result / total,
        "correct": correct,
        "syntax_errors": syntax_errors,
        "wrong_results": wrong_result,
        "total": total,
    }


results = evaluate(predictions, test_examples[:N_EXAMPLES])

print("=" * 50)
print("RAG BASELINE RESULTS")
print("=" * 50)
print(f"Execution accuracy: {results['execution_accuracy']:.1%} ({results['correct']}/{results['total']})")
print(f"Syntax error rate:  {results['syntax_error_rate']:.1%} ({results['syntax_errors']}/{results['total']})")
print(f"Wrong result rate:  {results['wrong_result_rate']:.1%} ({results['wrong_results']}/{results['total']})")

## 7. Compare All Approaches

In [ ]:
print("\n" + "=" * 70)
print("COMPARISON: ALL APPROACHES")
print("=" * 70)
print(f"{'Approach':<40} {'Exec Acc':>10} {'Syntax Err':>12}")
print("-" * 70)
print(f"{'Base model (zero-shot)':<40} {'37.2%':>10} {'21.8%':>12}")
print(f"{'RAG baseline (3-shot retrieval)':<40} {results['execution_accuracy']:>10.1%} {results['syntax_error_rate']:>12.1%}")
print(f"{'Fine-tuned v1 (chat template fix)':<40} {'51.8%':>10} {'5.4%':>12}")
print(f"{'Fine-tuned v1 + COLLATE NOCASE':<40} {'68.2%':>10} {'5.4%':>12}")
print("=" * 70)

## 8. Save Results

In [ ]:
import os
os.makedirs("results", exist_ok=True)

rag_save = {
    "experiment": "RAG baseline (3-shot retrieval)",
    "model": MODEL_ID,
    "adapter": None,
    "method": "RAG few-shot",
    "retrieval_model": "all-MiniLM-L6-v2",
    "top_k": 3,
    "prompt_format": "few-shot with retrieved examples",
    "generation_config": {k: str(v) for k, v in GENERATION_CONFIG.items()},
    "n_examples": N_EXAMPLES,
    "execution_accuracy": results["execution_accuracy"],
    "syntax_error_rate": results["syntax_error_rate"],
    "wrong_result_rate": results["wrong_result_rate"],
    "correct": results["correct"],
    "syntax_errors": results["syntax_errors"],
    "wrong_results": results["wrong_results"],
    "total_time_seconds": round(total_time, 1),
    "predictions": predictions,
}

with open("results/rag_results.json", "w") as f:
    json.dump(rag_save, f, indent=2)

print("Saved to results/rag_results.json")

In [ ]:
# Download results to your local machine
from google.colab import files
files.download("results/rag_results.json")

## 9. Sample Predictions

Inspect a few predictions to see how the RAG approach works.

In [ ]:
# Show 10 sample predictions
for i in range(10):
    ex = test_examples[i]
    gold = build_sql_string(ex["sql"], ex["table"]["header"], ex["table"]["types"])
    pred = predictions[i]
    
    # Check correctness
    pred_res, pred_err = execute_sql(ex["table"], pred)
    gold_res, _ = execute_sql(ex["table"], gold)
    status = "SYNTAX_ERR" if pred_err else ("CORRECT" if normalize_result(pred_res) == normalize_result(gold_res) else "WRONG")
    emoji = {"CORRECT": "✅", "WRONG": "❌", "SYNTAX_ERR": "⚠️"}[status]
    
    print(f"{emoji} Example {i}")
    print(f"   Q: {ex['question']}")
    print(f"   Gold: {gold}")
    print(f"   Pred: {pred}")
    print()